In [ ]:
import pandas as pd

In [ ]:
url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet"
columns = ['PULocationID', 'DOLocationID', 'trip_distance', 'total_amount', 'tpep_pickup_datetime']
df = pd.read_parquet(url, columns=columns).head(1000)
df.head()

In [ ]:
df.shape

In [ ]:
row = df.iloc[0]
row

In [ ]:
import json
from kafka import KafkaProducer
import dataclasses

# def json_serializer(data):
#     return json.dumps(data).encode('utf-8')

def ride_serializer(ride):
    ride_dict = dataclasses.asdict(ride)
    json_str = json.dumps(ride_dict)
    return json_str.encode('utf-8')


server = 'localhost:9092'

# producer = KafkaProducer(
#     bootstrap_servers=[server],
#     value_serializer=json_serializer
# )

producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=ride_serializer
)

In [ ]:
from models import Ride, ride_deserializer, ride_from_row

ride = ride_from_row(df.iloc[1])
ride

In [ ]:
ride_serializer(ride)

In [ ]:
# import dataclasses

topic_name = 'rides'

# producer.send(topic_name, value=dataclasses.asdict(ride))
# producer.flush()

producer.send(topic_name, value=ride)
producer.flush()

In [ ]:
import time

t0 = time.time()

for _, row in df.iterrows():
    ride = ride_from_row(row)
    producer.send(topic_name, value=ride)
    print(f"Sent: {ride}")
    time.sleep(0.01)

producer.flush()

t1 = time.time()
print(f'took {(t1 - t0):.2f} seconds')